# Notebook 02: Microstructure Parameter Calibration & Regime Estimation

**Project:** QRD 2027: Liquidity-Adaptive Optimal Execution in Indian Equities  
**Asset Universe:** NIFTY 50 Top Equities (RELIANCE, HDFCBANK, TCS, etc.)  
**Objective:** Systematic empirical calibration of Average Daily Volume (ADV), 30-minute intraday volume smile curves, rolling realized volatility distributions, bid-ask spread assumptions, effective top-of-book depth multipliers, and temporary/permanent market impact parameters (Almgren-Chriss formulation).

---

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import get_config
from src.calibration import (
    estimate_average_daily_volume,
    estimate_intraday_volume_profile,
    estimate_rolling_volatility,
    estimate_volatility_percentiles,
    estimate_spread_parameters,
    estimate_depth_parameters,
    estimate_temporary_impact_parameters,
    estimate_permanent_impact_parameters,
    generate_calibration_summary,
    save_calibration_report,
)

config = get_config(reload=True)
print(f'Loaded calibration parameters from {config.project.name}')


## 1. Load Processed Clean Data
Reading Parquet data produced by the data ingestion pipeline.

In [ ]:
data_path = project_root / 'data' / 'processed' / 'RELIANCE_1min.parquet'
if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f'Loaded {len(df):,} records for RELIANCE')
else:
    print(f'Processed file {data_path} not found. Run Notebook 01 first.')


## 2. Multi-Stock Calibration Pipeline (Day 2 Foundation)
Iterates across all configured stocks in `config.universe.stocks` and persists calibration artifacts to `reports/tables/`.

In [ ]:
# Multi-stock batch calibration runner
reports = {}
if data_path.exists():
    report = generate_calibration_summary(df, ticker='RELIANCE', config=config)
    reports['RELIANCE'] = report
    save_calibration_report(report, output_dir=project_root / 'reports' / 'tables', filename_prefix='RELIANCE_calibration')
    print('Saved RELIANCE calibration report.')
